In [3]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

print("Token loaded:", token is not None)

Token loaded: True


In [4]:
import os

os.environ["GITHUB_TOKEN"] = token

!git clone https://$GITHUB_TOKEN@github.com/nehnamehranmk638-dev/multilingual-rag-research.git

Cloning into 'multilingual-rag-research'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 108 (delta 46), reused 70 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 862.59 KiB | 12.32 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [5]:
%cd /content/multilingual-rag-research

/content/multilingual-rag-research


In [6]:
!git config --global credential.helper store

In [7]:
import subprocess

username = "nehnamehrankmk638-dev"

credential = f"""protocol=https
host=github.com
username={username}
password={token}

"""

subprocess.run(
    ["git", "credential", "approve"],
    input=credential,
    text=True,
    check=True
)

print("GitHub authentication configured.")

GitHub authentication configured.


In [8]:
!git fetch origin
!git switch nehna

branch 'nehna' set up to track 'origin/nehna'.
Switched to a new branch 'nehna'


In [10]:
import json
import re

with open("data/corpus_ml.json", "r", encoding="utf-8") as f:
    corpus_ml = json.load(f)

with open("data/questions_ml.json", "r", encoding="utf-8") as f:
    questions_ml = json.load(f)

print("Corpus:", len(corpus_ml))
print("Questions:", len(questions_ml))

Corpus: 247
Questions: 100


In [11]:
def tokenize_ml(text):
    tokens = re.findall(r"[\u0D00-\u0D7F]+", text)
    return tokens

print(tokenize_ml(corpus_ml[0]["text"][:100]))

['ഹരപ്പയുമായി', 'യാതൊരു', 'ബന്ധവും', 'കാണുന്ന', 'തരത്തിലല്ല', 'അവയുടെ', 'രീതി', 'ഹരപ്പൻ', 'കോട്ടക്ക്', 'തെക്ക്', 'ഭാഗത്തായി', 'കണ്ട']


In [12]:
!pip install -q rank_bm25

In [13]:
from rank_bm25 import BM25Okapi

tokenized_corpus_ml = [
    tokenize_ml(doc["text"])
    for doc in corpus_ml
]

bm25_ml = BM25Okapi(tokenized_corpus_ml)

print("BM25 index built successfully!")
print("Number of documents:", len(tokenized_corpus_ml))

BM25 index built successfully!
Number of documents: 247


In [14]:
def bm25_search_ml(query, k=5):
    query_tokens = tokenize_ml(query)

    scores = bm25_ml.get_scores(query_tokens)

    ranked_ids = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )

    return ranked_ids[:k]

In [15]:
q = questions_ml[0]

print("Question:", q["question"])
print("Gold id:", q["gold_passage_id"])
print("Top 5:", bm25_search_ml(q["question"], k=5))

Question: ഭരതനാട്യത്തിൽ രണ്ട് കൈകളും കാണിക്കുന്ന സംയുക്ത മുദ്രകൾ  എത്ര എണ്ണമുണ്ട് ?
Gold id: 49
Top 5: [49, 232, 237, 144, 55]


In [16]:
def recall_at_k(questions, results, k):
    """
    Recall@k:
    Percentage of questions where the gold passage
    appears in the top-k retrieved passages.
    """
    hits = 0

    for q in questions:
        question_id = q["question_id"]
        gold_id = q["gold_passage_id"]

        retrieved_ids = results[question_id][:k]

        if gold_id in retrieved_ids:
            hits += 1

    return hits / len(questions)

In [17]:
def mrr(questions, results):
    """
    Mean Reciprocal Rank:
    Average of 1/rank of the gold passage.
    If the gold passage is not retrieved, contribution = 0.
    """
    reciprocal_ranks = []

    for q in questions:
        question_id = q["question_id"]
        gold_id = q["gold_passage_id"]

        retrieved_ids = results[question_id]

        if gold_id in retrieved_ids:
            rank = retrieved_ids.index(gold_id) + 1
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)

    return sum(reciprocal_ranks) / len(reciprocal_ranks)

In [18]:
bm25_results_ml = {}

for q in questions_ml:
    bm25_results_ml[q["question_id"]] = bm25_search_ml(
        q["question"],
        k=10
    )

print("BM25 retrieval completed!")
print("Questions evaluated:", len(bm25_results_ml))

BM25 retrieval completed!
Questions evaluated: 100


In [19]:
for k in [1, 3, 5, 10]:
    print(
        f"Malayalam BM25 Recall@{k}:",
        recall_at_k(questions_ml, bm25_results_ml, k)
    )

print(
    "Malayalam BM25 MRR:",
    mrr(questions_ml, bm25_results_ml)
)

Malayalam BM25 Recall@1: 0.65
Malayalam BM25 Recall@3: 0.75
Malayalam BM25 Recall@5: 0.82
Malayalam BM25 Recall@10: 0.89
Malayalam BM25 MRR: 0.7151904761904762


In [20]:
with open("results/bm25_top10_ml.json", "w", encoding="utf-8") as f:
    json.dump(bm25_results_ml, f, ensure_ascii=False, indent=2)

print("Saved: results/bm25_top10_ml.json")

Saved: results/bm25_top10_ml.json


In [21]:
import os

print(
    "File exists:",
    os.path.exists("results/bm25_top10_ml.json")
)

File exists: True


In [22]:
import json

with open("results/bm25_top10_ml.json", "r", encoding="utf-8") as f:
    bm25_results_ml = json.load(f)

print("Loaded BM25 results:", len(bm25_results_ml))

Loaded BM25 results: 100


In [25]:
failed_queries_ml = []

for q in questions_ml:
    question_id = q["question_id"]
    gold_id = q["gold_passage_id"]

    # JSON converts dictionary keys to strings
    retrieved_ids = bm25_results_ml[str(question_id)]

    if gold_id not in retrieved_ids:
        failed_queries_ml.append(q)

print("Total failed queries:", len(failed_queries_ml))

Total failed queries: 11


In [26]:
for i, q in enumerate(failed_queries_ml, 1):

    question_id = q["question_id"]
    gold_id = q["gold_passage_id"]
    retrieved_ids = bm25_results_ml[str(question_id)]

    print("=" * 100)
    print(f"FAILURE {i}")
    print("Question:", q["question"])
    print("Gold passage ID:", gold_id)
    print("Retrieved top 10:", retrieved_ids)

    print("\nGOLD PASSAGE:")
    print(corpus_ml[gold_id]["text"])

    print("\nTOP RETRIEVED PASSAGE:")
    print(corpus_ml[retrieved_ids[0]]["text"])

    print()

FAILURE 1
Question: നവോത്ഥാന നായകന്മാരായ ശ്രീനാരായണ ഗുരുവിന്റെയും അയ്യങ്കാളിയുടെയും കേന്ദ്രം  ഏതായിരുന്നു ?
Gold passage ID: 44
Retrieved top 10: [118, 96, 45, 26, 66, 0, 1, 2, 3, 4]

GOLD PASSAGE:
അദ്ദേഹം പുതിയ ചന്തകൾ നിർമ്മിക്കുകയും തമിഴ്‌നാട്ടിലെ മദ്രാസ്, തിരുനൽവേലി എന്നിവിടങ്ങളിൽ നിന്നുള്ള കച്ചവടക്കാരെ കൊല്ലത്ത് വ്യാപാരത്തിനായി ക്ഷണിക്കുകയും ചെയ്തു. ഇതേത്തുടർന്ന് കശുവണ്ടി, കയർ, സുഗന്ധവ്യഞ്ജനങ്ങൾ എന്നിവയുടെ കച്ചവടം കൊല്ലത്ത് തഴച്ചു. ഇക്കാലയളവിലെ കൊല്ലത്തിന്റെ മേന്മകണ്ടാണ് കൊല്ലം കണ്ടവന് ഇല്ലം വേണ്ട എന്ന ചൊല്ല് ഉണ്ടായത്. 1811 റസിഡൻറ് മൺറോയ്ക്കുവേണ്ടി പണിയിപ്പിച്ചതാണ് ആശ്രാമം എന്ന സ്ഥലത്തെ കൊല്ലം റസിഡൻസി. ആതർ എന്ന എൻജിനീയർ ആണ് ഇതിന് നേതൃത്വം കൊടുത്തത്. റസിഡൻറിന്റെ ആസ്ഥാനം, ദിവാൻ കച്ചേരി, അപ്പീൽകോടതി തുടങ്ങിയവയെല്ലാം ആദ്യം കൊല്ലത്തായിരുന്നു. 1803 മുതൽ 1830 വരെ ഇംഗ്ലീഷ് പട്ടാളം തമ്പടിച്ചിരുന്നത് കൊല്ലം കൻറോൺമെൻറിലാണ്. 1809-ൽ ബ്രിട്ടീഷ് ഈസ്റ്റ് ഇന്ത്യാ കമ്പനിയും തിരിവിതാംകൂറും തമ്മിൽ  കൊല്ലം യുദ്ധം നടന്നു. സ്വാതി തിരുനാളിന്റെ കാലത്തോടെയാണ് ദിവാൻ കച്ചേരി തലസ്ഥാനത്തേക്ക് മാറ്റിയത്. തിരുവനന

## BM25 Error Analysis

Malayalam BM25 was evaluated on 100 QA questions.

### Retrieval Performance

- Recall@1: 0.65
- Recall@3: 0.75
- Recall@5: 0.82
- Recall@10: 0.89
- MRR: 0.7152

BM25 failed to retrieve the gold passage within the top 10 results for 11 questions.

### Observed Failure Patterns

The failed cases mainly represent lexical retrieval limitations. Several questions require matching concepts expressed with different wording or linguistic forms in Malayalam. In some cases, BM25 retrieved passages from the same broad topic but not the annotated gold passage.

One example is the question about Gurjara-Pratiharas, where the gold passage explicitly contains the answer but the retrieved passages discuss unrelated historical topics.

Another example is the Bangalore population-rank question, where the gold passage contains the answer "27th", but BM25 retrieved a general passage about India's geography.

One case also revealed a possible dataset annotation issue: the question uses wording corresponding to "bribe", while the gold passage describes Sri Lankan kings as paying tribute to Samudragupta. This case is flagged for later human review.

Overall, the failures indicate that lexical BM25 retrieval has limitations for Malayalam QA, particularly when the question and relevant passage use different wording.